In [1]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [4]:
todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [7]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [8]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [9]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [10]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [11]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [12]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [15]:
loop(messages)

Todo #1: Define variables, reference time axis, and interpret direction/initial separation.
Todo #2: Estimate/assume the Boston–New York distance since it is not provided (choose a reasonable value).
Todo #3: Set up relative motion equation with the 1-hour head start and solve for meeting time.
Todo #4: Compute clock time of meeting and optionally distance from a reference city.
Todo #5: Present final answer clearly.

Setup
- Let t = hours after 2:00 pm.
- Boston train speed: 60 mph toward New York.
- New York train leaves at 3:00 pm, i.e., at t = 1, speed 80 mph toward Boston.
- Let D = distance between Boston and New York along the track.

Todo #1: Define variables, reference time axis, and interpret direction/initial separation.
Todo #2: Estimate/assume the Boston–New York distance since it is not provided (choose a reasonable value).
Todo #3: Set up relative motion equation with the 1-hour head start and solve for meeting time.
Todo #4: Compute clock time of meeting and optionally distance from a reference city.
Todo #5: Present final answer clearly.

Assumption (needed because distance not given)
Use a reasonable BostonNew York rail/road distance of about 215 miles (common approximation; actual rail distance 
varies by route).

Todo #1: Define variables, reference time axis, and interpret direction/initial separation.
Todo #2: Estimate/assume the Boston–New York distance since it is not provided (choose a reasonable value).
Todo #3: Set up relative motion equation with the 1-hour head start and solve for meeting time.
Todo #4: Compute clock time of meeting and optionally distance from a reference city.
Todo #5: Present final answer clearly.

...

Convert to clock time (+ distances)
- 0.107 hours ≈ 0.107×60 ≈ 6.43 minutes.
- Meeting time: 2:00 pm + 2 hours 6.4 min ≈ 4:06 pm.
Optional location:
- From Boston: 60t = 60(2.107) ≈ 126.4 miles
- From New York: D − 126.4 ≈ 215 − 126.4 = 88.6 miles.

Todo #1: Define variables, reference time axis, and interpret direction/initial separation.
Todo #2: Estimate/assume the Boston–New York distance since it is not provided (choose a reasonable value).
Todo #3: Set up relative motion equation with the 1-hour head start and solve for meeting time.
Todo #4: Compute clock time of meeting and optionally distance from a reference city.
Todo #5: Present final answer clearly.

Final
State meeting time based on the assumed BostonNew York distance and provide the general formula as well.

Todo #1: Define variables, reference time axis, and interpret direction/initial separation.
Todo #2: Estimate/assume the Boston–New York distance since it is not provided (choose a reasonable value).
Todo #3: Set up relative motion equation with the 1-hour head start and solve for meeting time.
Todo #4: Compute clock time of meeting and optionally distance from a reference city.
Todo #5: Present final answer clearly.

Let D be the Boston–New York distance (miles).  
By 3:00 pm, the Boston train has a 1-hour head start: 60 miles.

After 3:00 pm, they approach each other at a combined (relative) speed of  
60 + 80 = 140 mph.

So the time after 3:00 pm until they meet is
(D − 60)/140 hours, and the meeting time is
3:00 pm + (D − 60)/140 hours.

Because D isn’t given, assume a reasonable D ≈ 215 miles. Then:
- Time after 3:00 pm: (215 − 60)/140 = 155/140 ≈ 1.107 hours ≈ 1 hour 6.4 min  
- Meeting time ≈ 4:06 pm

(With this assumption, they meet about 126 miles from Boston, ~89 miles from New York.)